In [1]:
# Import dependencies
import openai
import instructor
from pydantic import BaseModel, Field
import os
import json
from typing import Any
from superlinked import framework as sl
from api.agents.superlinked_app.index import business_index, business
from api.agents.superlinked_app.query import query
from api.agents.superlinked_app.utils.utils import *

/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


00:40:35 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 648.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


00:40:36 superlinked.framework.common.space.embedding.model_based.embedding_engine_manager INFO   Consider caching model dimension.
00:40:36 superlinked.framework.dsl.index.index INFO   initialized index


### RAG Pipeline

In [2]:
client = instructor.from_openai(openai.OpenAI())

In [3]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")

In [4]:
qdrant_vdb = sl.QdrantVectorDatabase(
    url="http://localhost:6333",
    # Superlinked's QdrantVectorDatabase currently requires an api_key arg.
    # For local Qdrant this is typically unused, so we default to empty.
    api_key=os.getenv("QDRANT_API_KEY", ""),
)
parser = sl.DataFrameParser(business)

source_qdrant = sl.RestSource(
    business,
    parser=parser,
)

# RestExecutor needs sl.RestQuery (path for /api/v1/search/<query_path> by default).
business_rest_query = sl.RestQuery(
    rest_descriptor=sl.RestDescriptor(query_path="business_search"),
    query_descriptor=query,
)

executor_qdrant = sl.RestExecutor(
    sources=[source_qdrant],
    indices=[business_index],
    vector_database=qdrant_vdb,
    queries=[business_rest_query],
)
qdrant_app = executor_qdrant.run()


def Retrieve_context(question, qdrant_app, k=5):
    qdrant_results = qdrant_app.query(
    query,
    natural_query=question,
    limit=k,
)

    format_minute_columns_to_hhmm(sl.PandasConverter.to_pandas(qdrant_results))
    return qdrant_results


def _result_to_restaurants(result) -> list[dict[str, Any]]:
    df_columns = ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude", "stars", "review_count", "is_open", "categories", "attributes", "hours"]
    df = sl.PandasConverter.to_pandas(result).rename(columns={"id": "business_id"})
    # Derive `is_open` robustly.
    # Some payloads may contain only `is_open_i` (0/1) instead of the boolean `is_open`.
    if "is_open" not in df.columns:
        if "is_open_i" in df.columns:
            df["is_open"] = df["is_open_i"].astype(int)
        else:
            df["is_open"] = 0

    df = df.assign(
        categories=df.get("category_tags", df.get("categories_text")),
        is_open=df["is_open"].astype(int),
    )

    # Parse attributes/hours when present.
    for c in ("attributes", "hours"):
        if c in df.columns:
            df[c] = df[c].map(
                lambda v: json.loads(v) if isinstance(v, str) and v.strip()
                else ({} if v in ("", None) else v)
            )
        else:
            df[c] = {}
    return df.reindex(columns=df_columns).to_dict(orient="records")


def build_prompt(preprocessed_context, question):
    prompt=f"""
    You are a yelp shopping assistant that can answer question about the restaurants.
    You will be given a question and a list of context.
    Instructions:
    - You need to answer questions based on the provided context only
    - Never use the word context and rfer to it as the available businesses or amenities
    - respond naturally and provide as much details as possible to the user request 
    - Refrain from using filter sush as is_open =True ...Rather say open today

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt


def generate_answer(prompt):
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role":"system", "content": prompt}],
        temperature=0,
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question):
    context=Retrieve_context(question, qdrant_app)
    preprocessed_context=_result_to_restaurants(context)
    prompt=build_prompt(preprocessed_context, question)
    answer=generate_answer(prompt)

    final_response={
        "datamodel":answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": [e.id for e in context.entries],
        "retrieved_restaurant_names": [e.fields.get("name") for e in context.entries],
        "similarity_score": [e.metadata.score for e in context.entries],
    }
    return final_response
    

/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:87: UserWarning: Api key is used with an insecure connection.
  self._client = AsyncQdrantClient(
/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:92: UserWarning: Api key is used with an insecure connection.
  self._sync_client = QdrantClient(


00:40:54 superlinked.framework.query.query_dag_evaluator INFO   initialized query dag
00:40:54 superlinked.framework.online.online_dag_evaluator INFO   initialized entity dag


In [10]:
output=rag_pipeline("What are the best restaurants in Tampa?")

00:43:54 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
00:43:54 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
00:43:55 superlinked.framework.dsl.executor.query.query_executor INFO   executed query


In [11]:
output["answer"]

"Some of the best restaurants in Tampa based on ratings and reviews include:\n\n1. Shells Seafood Restaurant - South Tampa: A highly rated seafood restaurant with 4.5 stars and 453 reviews. It offers a full bar, casual yet classy ambiance, good for kids, and serves lunch and dinner. It has free WiFi, happy hour, and is wheelchair accessible.\n\n2. Saigon Deli: A Vietnamese deli with 4.5 stars and 738 reviews. It has a casual and classy atmosphere, good for lunch, offers outdoor seating, and caters to groups. It does not serve alcohol but has bike parking and a parking lot.\n\n3. Felicitous: A sandwich and dessert spot with 4.5 stars and 273 reviews. It has a hipster, casual vibe, offers outdoor seating, caters, and is good for groups. It accepts credit cards and has free WiFi.\n\n4. Matoi Sushi: A Korean and Japanese sushi bar with 4 stars and 594 reviews. It offers beer and wine, has a quiet and casual ambiance, good for lunch and dinner, and provides delivery and reservations.\n\n5. 

### RAG Pipeline with grounded answer

In [14]:
class RAGUsedContext (BaseModel):
    id: str = Field(description="The id of the restaurants used to answer the question")
    description: str = Field(description="A short description of the business")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    references: list[RAGUsedContext]=Field(description="The context used to answer the question")

In [18]:
qdrant_vdb = sl.QdrantVectorDatabase(
    url="http://localhost:6333",
    # Superlinked's QdrantVectorDatabase currently requires an api_key arg.
    # For local Qdrant this is typically unused, so we default to empty.
    api_key=os.getenv("QDRANT_API_KEY", ""),
)
parser = sl.DataFrameParser(business)

source_qdrant = sl.RestSource(
    business,
    parser=parser,
)

# RestExecutor needs sl.RestQuery (path for /api/v1/search/<query_path> by default).
business_rest_query = sl.RestQuery(
    rest_descriptor=sl.RestDescriptor(query_path="business_search"),
    query_descriptor=query,
)

executor_qdrant = sl.RestExecutor(
    sources=[source_qdrant],
    indices=[business_index],
    vector_database=qdrant_vdb,
    queries=[business_rest_query],
)
qdrant_app = executor_qdrant.run()


def Retrieve_context(question, qdrant_app, k=5):
    qdrant_results = qdrant_app.query(
    query,
    natural_query=question,
    limit=k,
)

    format_minute_columns_to_hhmm(sl.PandasConverter.to_pandas(qdrant_results))
    return qdrant_results


def _result_to_restaurants(result) -> list[dict[str, Any]]:
    df_columns = ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude", "stars", "review_count", "is_open", "categories", "attributes", "hours"]
    df = sl.PandasConverter.to_pandas(result).rename(columns={"id": "business_id"})
    # Derive `is_open` robustly.
    # Some payloads may contain only `is_open_i` (0/1) instead of the boolean `is_open`.
    if "is_open" not in df.columns:
        if "is_open_i" in df.columns:
            df["is_open"] = df["is_open_i"].astype(int)
        else:
            df["is_open"] = 0

    df = df.assign(
        categories=df.get("category_tags", df.get("categories_text")),
        is_open=df["is_open"].astype(int),
    )

    # Parse attributes/hours when present.
    for c in ("attributes", "hours"):
        if c in df.columns:
            df[c] = df[c].map(
                lambda v: json.loads(v) if isinstance(v, str) and v.strip()
                else ({} if v in ("", None) else v)
            )
        else:
            df[c] = {}
    return df.reindex(columns=df_columns).to_dict(orient="records")


def build_prompt(preprocessed_context, question):
    prompt=f"""
    You are a yelp shopping assistant that can answer question about the restaurants.
    You will be given a question and a list of context.
    Instructions:
    - You need to answer questions based on the provided context only
    - Never use the word context and rfer to it as the available businesses or amenities
   - As an output you need to provide:

    * The answer to the question based on the provided context.
    * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
    * Short description (1-2 sentences) of the item based on the description provided in the context.

    - The short description should have the name of the restaurant.
    - The answer to the question should contain detailed information about the restaurant and returned with detailed specification in bullet points.


    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt


def generate_answer(prompt):
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role":"system", "content": prompt}],
        temperature=0,
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question):
    context=Retrieve_context(question, qdrant_app)
    preprocessed_context=_result_to_restaurants(context)
    prompt=build_prompt(preprocessed_context, question)
    answer=generate_answer(prompt)

    final_response={
        "datamodel":answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": [e.id for e in context.entries],
        "retrieved_restaurant_names": [e.fields.get("name") for e in context.entries],
        "similarity_score": [e.metadata.score for e in context.entries],
    }
    return final_response
    

/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:87: UserWarning: Api key is used with an insecure connection.
  self._client = AsyncQdrantClient(
/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:92: UserWarning: Api key is used with an insecure connection.
  self._sync_client = QdrantClient(


00:55:42 superlinked.framework.query.query_dag_evaluator INFO   initialized query dag
00:55:42 superlinked.framework.online.online_dag_evaluator INFO   initialized entity dag


In [20]:
output=rag_pipeline("What are the best restaurants in Tampa?")

00:56:02 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
00:56:02 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
00:56:02 superlinked.framework.dsl.executor.query.query_executor INFO   executed query


In [21]:
output

{'datamodel': RAGGenerationResponse(answer="Here are some of the best restaurants in Tampa based on ratings and reviews:\n\n1. Shells Seafood Restaurant - South Tampa\n- Rating: 4.5 stars from 453 reviews\n- Cuisine: Seafood\n- Features: Full bar, casual and classy ambiance, good for kids, happy hour, free WiFi, wheelchair accessible\n- Services: Delivery, takeout, table service, caters\n- Parking: Lot available\n- Hours: Open mostly afternoons and evenings\n\n2. Saigon Deli\n- Rating: 4.5 stars from 738 reviews\n- Cuisine: Vietnamese, deli, soup, coffee & tea\n- Features: No alcohol, casual and classy ambiance, good for kids, outdoor seating, has TV\n- Services: Delivery, caters\n- Parking: Lot available\n- Hours: Morning to late afternoon\n\n3. Felicitous\n- Rating: 4.5 stars from 273 reviews\n- Cuisine: Sandwiches, Cuban, vegetarian, desserts, coffee & tea\n- Features: No alcohol, casual and hipster ambiance, outdoor seating, dog friendly, free WiFi, wheelchair accessible\n- Service

In [22]:
print(output["answer"])

Here are some of the best restaurants in Tampa based on ratings and reviews:

1. Shells Seafood Restaurant - South Tampa
- Rating: 4.5 stars from 453 reviews
- Cuisine: Seafood
- Features: Full bar, casual and classy ambiance, good for kids, happy hour, free WiFi, wheelchair accessible
- Services: Delivery, takeout, table service, caters
- Parking: Lot available
- Hours: Open mostly afternoons and evenings

2. Saigon Deli
- Rating: 4.5 stars from 738 reviews
- Cuisine: Vietnamese, deli, soup, coffee & tea
- Features: No alcohol, casual and classy ambiance, good for kids, outdoor seating, has TV
- Services: Delivery, caters
- Parking: Lot available
- Hours: Morning to late afternoon

3. Felicitous
- Rating: 4.5 stars from 273 reviews
- Cuisine: Sandwiches, Cuban, vegetarian, desserts, coffee & tea
- Features: No alcohol, casual and hipster ambiance, outdoor seating, dog friendly, free WiFi, wheelchair accessible
- Services: Delivery, takeout, caters
- Parking: Street and lot available
-